# DeepSeek-OCR-2 server for Tamveen POS

Runtime → Change runtime type → **GPU** (T4 works, L4/A100 is faster). Then Runtime → Run all.
The last cell prints a `https://….trycloudflare.com` address: paste it into the POS under Settings → Scanning → OCR server.
The address changes every time the notebook restarts.

In [ ]:
!pip -q install --no-warn-conflicts transformers==4.46.3 tokenizers==0.20.3 einops easydict addict fastapi uvicorn python-multipart nest_asyncio
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

In [ ]:
import os, re, io, sys, time, glob, tempfile, contextlib
import torch
from transformers import AutoModel, AutoTokenizer

MODEL = 'deepseek-ai/DeepSeek-OCR-2'
gpu = torch.cuda.get_device_name(0)
# FlashAttention 2 needs an Ampere or newer GPU (L4, A100); a T4 uses plain attention.
attn = 'eager'
if torch.cuda.get_device_capability(0)[0] >= 8:
    try:
        import flash_attn  # noqa: F401
        attn = 'flash_attention_2'
    except ImportError:
        pass
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
model = AutoModel.from_pretrained(MODEL, _attn_implementation=attn, trust_remote_code=True, use_safetensors=True)
model = model.eval().cuda().to(torch.bfloat16)
print('Loaded on', gpu, 'with', attn)

In [ ]:
PROMPT = '<image>\nFree OCR. '

def clean(text):
    """Plain lines for the POS parser: drop grounding tags, turn table rows into lines."""
    text = re.sub(r'<\|ref\|>(.*?)<\|/ref\|>', r'\1', text or '')
    text = re.sub(r'<\|det\|>.*?<\|/det\|>', '', text)
    text = re.sub(r'</tr>|<br\s*/?>', '\n', text)
    text = re.sub(r'</t[dh]>', ' ', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = text.replace('|', ' ')
    return '\n'.join(re.sub(r'[ \t]+', ' ', l).strip() for l in text.splitlines() if l.strip())

def ocr(path):
    out = tempfile.mkdtemp()
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        try:
            res = model.infer(tokenizer, prompt=PROMPT, image_file=path, output_path=out, base_size=1024, image_size=768, crop_mode=True, save_results=True, eval_mode=True)
        except TypeError:  # older remote code without eval_mode
            res = model.infer(tokenizer, prompt=PROMPT, image_file=path, output_path=out, base_size=1024, image_size=768, crop_mode=True, save_results=True)
    if not isinstance(res, str) or not res.strip():
        files = glob.glob(os.path.join(out, '*.mmd')) + glob.glob(os.path.join(out, '*.md'))
        res = open(files[0], encoding='utf-8').read() if files else buf.getvalue()
    return res

# Quick check on a sample delivery note from the repo's test set.
!wget -q -O /content/D01.jpg https://raw.githubusercontent.com/sufyanansar/tamveen-pos/main/ocr-test/D01.jpg || true
if os.path.exists('/content/D01.jpg') and os.path.getsize('/content/D01.jpg') > 0:
    t0 = time.time(); print(clean(ocr('/content/D01.jpg'))); print(f'--- {time.time() - t0:.1f}s')

In [ ]:
import threading, subprocess, nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware

nest_asyncio.apply()
app = FastAPI()
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])
lock = threading.Lock()

@app.get('/')
def health():
    return {'model': MODEL, 'gpu': gpu}

@app.post('/ocr')
def read(file: UploadFile = File(...)):
    suffix = os.path.splitext(file.filename or '')[1] or '.jpg'
    with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
        f.write(file.file.read()); path = f.name
    t0 = time.time()
    with lock:  # one image at a time on the GPU
        raw = ocr(path)
    os.remove(path)
    text = clean(raw)
    print(time.strftime('%H:%M:%S'), 'scanned', file.filename, '->', len(text.splitlines()), f'lines in {time.time() - t0:.0f}s', flush=True)
    return {'engine': 'DeepSeek-OCR-2', 'secs': round(time.time() - t0, 1), 'text': text, 'raw': raw}

threading.Thread(target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'), daemon=True).start()
time.sleep(2)

tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        print('OCR server:', m.group(0)); break

In [ ]:
# Keeps the server running and shows each scan from the POS. Stop this cell (or the runtime) to shut it down.
while True:
    time.sleep(60)